# Phase 1: Smart Waste Classification - Dataset Exploration
This notebook inspects the waste dataset (TrashNet / Starter Dataset), class distributions, image properties, and analyzes class imbalance.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from src import config
from src.prepare_data import get_image_paths_and_labels

print("Project Root:", ROOT_DIR)
print("Dataset Classes:", config.CLASSES)

## 1. Class Distribution Analysis

In [ ]:
image_paths, labels = get_image_paths_and_labels(config.RAW_DATA_DIR)
print(f"Total images found: {len(image_paths)}")

class_counts = {}
for cls_name in config.CLASSES:
    idx = config.CLASS_TO_IDX[cls_name]
    cnt = labels.count(idx)
    class_counts[cls_name] = cnt
    print(f"  {cls_name:10s}: {cnt:4d} images ({cnt/max(1, len(labels))*100:5.1f}%)")

# Plot Distribution
plt.figure(figsize=(10, 4))
palette = ["#d97706", "#0284c7", "#475569", "#9333ea", "#16a34a", "#dc2626"]
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette=palette)
plt.title("Waste Dataset Class Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Image Count")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.show()

## 2. Visual Inspection: Sample Images per Category

In [ ]:
fig, axes = plt.subplots(1, len(config.CLASSES), figsize=(16, 3))

for idx, cls_name in enumerate(config.CLASSES):
    cls_dir = config.RAW_DATA_DIR / cls_name
    sample_file = next(cls_dir.glob("*.jpg"), next(cls_dir.glob("*.png"), None))
    if sample_file and sample_file.exists():
        img = Image.open(sample_file)
        axes[idx].imshow(img)
        axes[idx].set_title(f"{cls_name.capitalize()}\n({img.size[0]}x{img.size[1]})", fontsize=10, fontweight="bold")
    else:
        axes[idx].text(0.5, 0.5, "No Image", ha="center", va="center")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

## 3. Color Space & Channel Histogram Analysis

In [ ]:
if image_paths:
    sample_img = np.array(Image.open(image_paths[0]).convert("RGB"))
    fig, (ax_img, ax_hist) = plt.subplots(1, 2, figsize=(12, 4))
    ax_img.imshow(sample_img)
    ax_img.set_title("Sample Image")
    ax_img.axis("off")

    colors = ('r', 'g', 'b')
    for i, col in enumerate(colors):
        hist = np.histogram(sample_img[:, :, i], bins=256, range=[0, 256])[0]
        ax_hist.plot(hist, color=col, lw=2, label=f"{col.upper()} Channel")
    ax_hist.set_title("RGB Channel Histogram")
    ax_hist.set_xlabel("Pixel Intensity (0-255)")
    ax_hist.set_ylabel("Count")
    ax_hist.legend()
    ax_hist.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()